# Week 2 — Data Cleaning and Preprocessing

Reference: Brazilian E-Commerce Public Dataset by Olist.

Download the dataset from Kaggle and place the CSV files in `../data/raw/`.

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
orders.shape


In [ ]:
# Inspect structure and missing values
orders.info()
orders.isna().sum().sort_values(ascending=False)


In [ ]:
# Remove exact duplicates
orders = orders.drop_duplicates()

# Convert timestamps
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')


In [ ]:
# Feature engineering
orders['delivery_days'] = (
    orders['order_delivered_customer_date'] -
    orders['order_purchase_timestamp']
).dt.total_seconds() / 86400

orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date'] -
    orders['order_estimated_delivery_date']
).dt.total_seconds() / 86400

orders['is_late'] = orders['delivery_delay_days'].gt(0)
orders[['delivery_days', 'delivery_delay_days', 'is_late']].describe()


In [ ]:
# IQR outlier flag
valid = orders['delivery_days'].dropna()
q1, q3 = valid.quantile([0.25, 0.75])
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

orders['delivery_outlier'] = (
    orders['delivery_days'].lt(lower) |
    orders['delivery_days'].gt(upper)
)
orders['delivery_outlier'].value_counts(dropna=False)


In [ ]:
# Min-Max normalization
features = ['delivery_days', 'delivery_delay_days']
valid_index = orders[features].dropna().index
scaler = MinMaxScaler()
orders.loc[valid_index, features] = scaler.fit_transform(
    orders.loc[valid_index, features]
)


## Interpretation

Outliers should be investigated before removal. Missing delivery dates should be interpreted using order status. For machine-learning workflows, scaling parameters should be fitted on training data only.